In [2]:
import pandas as pd
from bs4 import BeautifulSoup

In [4]:
soup = BeautifulSoup(open("../../../data/drugbank/Drugbank_Fulldata.xml"),"xml")

In [5]:
drugs = soup.findAll("drug")

/tmp/ipykernel_1877189/522772564.py:1: DeprecationWarning: Call to deprecated method findAll. (Replaced by find_all) -- Deprecated since version 4.0.0.
  drugs = soup.findAll("drug")


In [9]:
data = []

for idx, drug in enumerate(drugs):
    print(f'{idx+1}/{len(drugs)}', end='\r')
    if len(drug.find_all(recursive=False)) <= 2:
        continue
    id_ = drug.find("drugbank-id").text
    name = drug.find("name").text
    
    data.append({
        'id': id_,
        'name': name,
    })
    
    general_infomation = [
        'volume-of-distribution',  # 分布容积
        'half-life',  # 半衰期
        'pharmacodynamics',  # 药效学
        'description',  # 描述
        'simple-description',  # 简单描述
        'clinical-description',  # 临床描述
        'state',  # 状态：固体/液体/气体
        'indication',  # 适应症
        'protein-binding',  # 蛋白质结合
        'mechanism-of-action',  # 作用机制
        'toxicity',  # 毒性
        'metabolism',  # 代谢
        'absorption',  # 吸收
        'route-of-elimination', # 消除途径
        'clearance',  # 清除率
    ]
    
    for v in general_infomation:
        if (ele:= drug.find(v)) is not None:
            data[-1][v] = ele.text.strip()
         
    sequences = [sequence.text.strip() for sequence in drug.find_all('sequence')]  # 可能包含多条肽链
    if len(sequences) > 0:   
        data[-1]['sequences'] = '|'.join(sequences)
    
    for prop in drug.find_all('property'):  # SMILES
        kind = prop.find('kind').text.strip()      
        if kind == 'SMILES':
            data[-1]['SMILES'] = prop.find('value').text.strip()
            break

In [10]:
df = pd.DataFrame(data)
df.to_csv('../../../data_feature/drugbank.csv', index=False)